In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/master.csv')
print(f"Loaded: {df.shape[0]:,} rows, {df.shape[1]} columns")

Loaded: 117,601 rows, 31 columns


In [2]:
# Convert date strings to proper datetime format
date_cols = [
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col])

print("Date columns fixed")

Date columns fixed


In [3]:
# Keep only delivered orders — we analyze completed transactions only
before = len(df)
df = df[df['order_status'] == 'delivered'].copy()
after = len(df)

print(f"Removed {before - after:,} non-delivered orders")
print(f"Remaining: {after:,} rows")

Removed 2,566 non-delivered orders
Remaining: 115,035 rows


In [4]:
# How many days did delivery actually take?
df['delivery_days'] = (
    df['order_delivered_customer_date'] -
    df['order_purchase_timestamp']
).dt.days

# Was the order delivered on time?
df['delivered_on_time'] = (
    df['order_delivered_customer_date'] <=
    df['order_estimated_delivery_date']
)

# Extract month and year for trend analysis
df['order_month'] = pd.to_datetime(df['order_purchase_timestamp']).dt.to_period('M')
df['order_year'] = pd.to_datetime(df['order_purchase_timestamp']).dt.year

print("New columns created")
print(df[['delivery_days', 'delivered_on_time', 'order_month', 'order_year']].head())

New columns created
   delivery_days  delivered_on_time order_month  order_year
0            8.0               True     2017-10        2017
1            8.0               True     2017-10        2017
2            8.0               True     2017-10        2017
3           13.0               True     2018-07        2018
4            9.0               True     2018-08        2018


In [5]:
# Fill missing English category names with Portuguese original
df['product_category_name_english'] = df['product_category_name_english'].fillna(
    df['product_category_name']
)

print(f"Missing categories remaining: {df['product_category_name_english'].isnull().sum()}")

Missing categories remaining: 1628


In [6]:
df.to_csv('../data/clean.csv', index=False)
print(f"Clean data saved: {df.shape[0]:,} rows, {df.shape[1]} columns")

Clean data saved: 115,035 rows, 35 columns


In [7]:
df['on_time_flag'] = df['delivered_on_time'].astype(int)
df.to_csv('../data/clean.csv', index=False)
print("Saved with on_time_flag column")

Saved with on_time_flag column
